# Chapter 5: Multi-Token Prediction and FP8 Quantization

We have now established the core architectural pillars of the DeepSeek model: Multi-Head Latent Attention and Mixture-of-Experts. These innovations define *what* the model computes. Now, we turn our attention to an equally important topic that defines *how* these computations are performed with incredible efficiency.

This chapter covers two key techniques central to DeepSeek's training methodology:

1. **Multi-Token Prediction (MTP)** — A technique that provides stronger training signals by predicting multiple future tokens simultaneously
2. **FP8 Quantization** — An advanced quantization framework that trades precision for significant gains in speed and memory

This chapter is divided into two main parts. First, we will dive deep into MTP, understanding its motivation, advantages, and exactly how DeepSeek implemented their advanced, causal version of it. After mastering MTP, we will move to the second part, a deep dive into the FP8 Quantization framework.

## Required Libraries

Let's begin by importing the necessary Python libraries:

- `math`: For mathematical operations
- `typing.Optional`: For type hints with optional parameters
- `torch`: PyTorch library for tensor operations and neural networks
- `torch.nn`: Neural network modules and functions
- `torch.nn.functional`: Functional interface for neural network operations

In [ ]:
import math
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

---

# Part 1: Multi-Token Prediction (MTP)

## From Single-Token to Multi-Token Prediction

The entire training process for the language models we've discussed so far has been based on a single, simple objective: **Next-Token Prediction**. In the standard approach, the model's goal is to predict the single token that immediately follows each input token.

**Multi-Token Prediction** changes this fundamental objective. Instead of predicting only the single next token, the model is trained to predict **multiple future tokens at once**. For example, from the input token "is," it might predict the sequence "transforming," "the," "world."

### The Four Key Advantages of MTP

1. **Densification of training signals** — Each training sample contains much more information for the model to learn from, as it receives gradient signals across multiple future steps simultaneously.

2. **Improved data efficiency** — Since each training sample is more informative, the model achieves higher performance with the same amount of training data. The original MTP paper demonstrated this on coding benchmarks like MBPP and HumanEval.

3. **Better planning by prioritizing "choice points"** — The MTP loss implicitly assigns higher weights to critical tokens that significantly influence the future outcome, forcing the model to develop better internal representations for planning.

4. **Higher inference speed via speculative decoding** — MTP heads can serve as a "draft" model for speculative decoding, achieving up to 3x speedups on certain tasks.

## The DeepSeek MTP Architecture

While the original MTP paper from Meta proved the concept's effectiveness using **independent** output heads, DeepSeek recognized a key opportunity for improvement. Their implementation is designed to **sequentially** predict additional tokens and keep the complete causal chain at each prediction depth.

### Key Design Principles:

1. **Starting Point**: The Shared Transformer Trunk processes input tokens to produce the initial hidden states $h^0$
2. **Sequential Chain**: A series of MTP Modules are chained together, each producing a refined hidden state for the next
3. **Each MTP Module**:
   - Takes the hidden state from the previous step ($h^{k-1}_i$) and the embedding of the future token ($\text{Emb}(t_{i+k})$)
   - Normalizes both via RMSNorm, concatenates, and projects back to model dimension
   - Passes through a dedicated Transformer block
   - Outputs a refined hidden state that serves dual purpose: passed to next head AND projected to vocabulary logits

### The Key Equations:

**Equation 5.1** — Merge and project:
$$h'^k_i = M_k[\text{RMSNorm}(h^{k-1}_i) \; ; \; \text{RMSNorm}(\text{Emb}(t_{i+k}))]$$

**Equation 5.2** — Transformer refinement:
$$h^k_i = \text{TRM}_k(h'^k_i)$$

**Equation 5.3** — Token prediction:
$$P^k_{i+k+1} = \text{OutHead}(h^k_i)$$

The total loss is the sum of the main next-token prediction loss and the weighted average of all MTP losses:
$$\mathcal{L} = \mathcal{L}_{\text{main}} + \frac{\lambda}{D} \sum_{k=1}^{D} \mathcal{L}^k_{\text{MTP}}$$

## Implementing a Causal Multi-Token Prediction Module from Scratch

We will build the entire MTP system in stages:

1. **RMSNorm**: The normalization layer used throughout the DeepSeek architecture
2. **The MTP Module**: The core component that processes one step in the causal chain
3. **The Main Model**: A wrapper class that integrates the main Transformer trunk and the chain of MTP modules
4. **The Forward Pass & Loss**: The complete logic for sequential prediction and the combined loss calculation

### Listing 5.1: RMSNorm

We begin by defining RMSNorm (Root Mean Square Layer Normalization), the specific normalization layer used throughout the DeepSeek architecture. This is a foundational utility that ensures numerical stability during training.

In [ ]:
class RMSNorm(nn.Module):
    """
    Implements Root Mean Square Layer Normalization.
    """
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Calculate the inverse square root of the mean of squares
        norm_x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        # Apply the learnable weight
        return self.weight * norm_x

### Listing 5.2: The Causal MTP Module

Next, we build the core of the MTP architecture: the `DeepSeekMTPModule`. It contains a single, dedicated Transformer block and the necessary projection layers. Its purpose is to take the hidden state from the previous step and the embedding of the next token, and produce a refined hidden state for the next step in the causal chain.

This class is a direct implementation of the logic from Equations 5.1 and 5.2:

1. **Normalize** both the previous hidden state and the future token embedding via separate RMSNorm layers
2. **Concatenate** them to form a merged embedding of dimension $2d$
3. **Project** back down to dimension $d$ via the projection matrix $M_k$
4. **Refine** through a dedicated Transformer block to produce the new hidden state

In [ ]:
class DeepSeekMTPModule(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int, dropout: float = 0.0):
        super().__init__()
        self.d_model = d_model

        self.projection_matrix = nn.Linear(2 * d_model, d_model, bias=False)

        self.transformer_block = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )

        self.norm_hidden = RMSNorm(d_model)
        self.norm_embed = RMSNorm(d_model)

    def forward(self, h_prev: torch.Tensor, future_token_embeds: torch.Tensor) -> torch.Tensor:
        h_normed = self.norm_hidden(h_prev)
        embed_normed = self.norm_embed(future_token_embeds)

        concatenated = torch.cat([h_normed, embed_normed], dim=-1)
        h_prime = self.projection_matrix(concatenated)

        h_output = self.transformer_block(h_prime)

        return h_output

### Listings 5.3 & 5.4: The Full MTP Model Architecture with Forward Pass

Now that we have our `DeepSeekMTPModule` building block, we can assemble the full model. Pay close attention to how the different components are organized:

- A **shared embedding** layer (`shared_embed`) used by both the main trunk and all MTP modules
- A **shared output head** (`shared_lm_head`) with **weight tying** to the embedding layer
- The main **Transformer backbone** (Shared Transformer Trunk)
- A **list of MTP modules**, one for each prediction depth

The `forward` method implements the complete sequential MTP process:

1. Input tokens pass through the main Transformer trunk to produce `h_main`
2. A loop iterates through each MTP module in the chain
3. For each depth $k$, the hidden state from the previous step and the ground-truth embedding of the $k$-th future token generate the new hidden state
4. The logits for each future token are computed from the refined hidden state
5. The total loss combines the main next-token prediction loss with the weighted average of MTP losses:

$$\mathcal{L} = \mathcal{L}_{\text{main}} + \frac{\lambda}{D} \sum_{k=1}^{D} \mathcal{L}^k_{\text{MTP}}$$

In [ ]:
class DeepSeekV3WithMTP(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int,
        num_layers: int,
        nhead: int,
        num_mtp_heads: int,     # D (number of MTP depths)
        dim_feedforward: int,
        dropout: float = 0.0,
        mtp_loss_weight: float = 0.1
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.nhead = nhead
        self.num_mtp_heads = num_mtp_heads
        self.dim_feedforward = dim_feedforward
        self.dropout = dropout
        self.mtp_loss_weight = mtp_loss_weight

        # Shared components used across the model
        self.shared_embed = nn.Embedding(vocab_size, d_model)
        self.shared_lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # Main transformer backbone (Shared Transformer Trunk)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
                dropout=dropout, activation='gelu', batch_first=True,
                norm_first=True
            )
            for _ in range(num_layers)
        ])
        self.norm_f = RMSNorm(d_model)

        # Weight tying between embedding and output head
        self.shared_lm_head.weight = self.shared_embed.weight

        # The chain of MTP modules
        self.mtp_modules = nn.ModuleList([
            DeepSeekMTPModule(d_model, nhead, dim_feedforward, dropout)
            for _ in range(num_mtp_heads)
        ])

    def get_embedding(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Shared embedding lookup used by the main trunk and all MTP modules."""
        return self.shared_embed(input_ids)

    def get_output_logits(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """Shared un-embedding matrix that projects hidden states to vocabulary logits."""
        return self.shared_lm_head(hidden_states)

    def forward(self, input_ids: torch.Tensor, targets: Optional[torch.Tensor] = None):
        B, S = input_ids.shape

        # --- Main model forward pass ---
        x = self.get_embedding(input_ids)
        for block in self.blocks:
            x = block(x)
        h_main = self.norm_f(x)
        logits_main = self.get_output_logits(h_main)

        all_logits = [logits_main]
        h_prev = h_main

        # --- MTP chain: Sequential prediction ---
        for depth_k in range(1, self.num_mtp_heads + 1):
            L = S - depth_k
            if L <= 0: break

            h_prev_sliced = h_prev[:, :L, :]

            future_token_ids = input_ids[:, depth_k:depth_k + L]
            future_token_embeds = self.get_embedding(future_token_ids)

            h_curr = self.mtp_modules[depth_k - 1](h_prev_sliced, future_token_embeds)

            logits_k = self.get_output_logits(h_curr)
            all_logits.append(logits_k)

            h_prev = h_curr

        # --- Loss computation ---
        loss = None
        if targets is not None:
            # Main model loss (standard next-token prediction)
            shift_logits = logits_main[:, :-1, :].contiguous()
            shift_targets = targets[:, 1:].contiguous()
            total_loss = F.cross_entropy(
                shift_logits.view(-1, self.vocab_size),
                shift_targets.view(-1)
            )

            mtp_loss_sum = 0.0
            for k, logits_k in enumerate(all_logits[1:], start=1):
                # MTP head k at position i predicts token at position i+k+1
                mtp_logits = logits_k[:, :-1, :].contiguous()
                mtp_targets = targets[:, k + 1:k + 1 + mtp_logits.size(1)].contiguous()
                if mtp_targets.numel() == 0:
                    continue
                loss_mtp_k = F.cross_entropy(
                    mtp_logits.view(-1, self.vocab_size),
                    mtp_targets.view(-1)
                )
                mtp_loss_sum += loss_mtp_k

            # Final loss: L = L_main + (lambda/D) * sum(L_MTP^k)
            if self.num_mtp_heads > 0 and mtp_loss_sum > 0:
                mtp_loss_weighted = (self.mtp_loss_weight / self.num_mtp_heads) * mtp_loss_sum
                total_loss += mtp_loss_weighted

            loss = total_loss

        return {"logits_all": all_logits, "loss": loss}

### Understanding the MTP Forward Pass

The forward method above contains the complete MTP logic. Let's trace through what happens:

**Main Trunk:**
- Input tokens are embedded and passed through the stack of Transformer blocks
- The output `h_main` (Hidden State 0, or $h^0$) is our starting point
- The main logits predict the immediate next token at each position

**MTP Chain:**
- For each depth `k` from 1 to D:
  - Slice `h_prev` to length `S - k` (accounting for the causal offset)
  - Get the ground-truth embeddings for the `k`-th future tokens
  - Pass both through the `k`-th MTP module to get refined hidden state `h_curr`
  - Compute logits from `h_curr` using the **shared** output head
  - `h_curr` becomes `h_prev` for the next depth — this is the **causal chain**

**Loss Calculation:**
- The main loss uses standard next-token prediction (shifted by 1)
- Each MTP head's loss is computed for its corresponding future token target
- The final loss combines them: $\mathcal{L} = \mathcal{L}_{\text{main}} + \frac{\lambda}{D} \sum_k \mathcal{L}^k_{\text{MTP}}$

Notice how the sequence length of the logit tensors **decreases by one at each step**, confirming the causal chain is correctly implemented.

### Listing 5.5: Verifying the Causal MTP Implementation

Let's verify our implementation by creating a model with:
- `vocab_size = 1000`
- `d_model = 128` with `nhead = 8`
- `num_layers = 6` Transformer blocks
- `num_mtp_heads = 3` (D=3, predict next 3 tokens)

We'll pass a batch of random token IDs through the model and verify the output shapes.

In [ ]:
def verify_deepseek_v3_mtp():
    # --- Model configuration ---
    vocab_size = 1000
    d_model = 128
    num_layers = 6
    nhead = 8
    num_mtp_heads = 3  # D=3 (predict next 3 tokens)
    dim_feedforward = 512
    mtp_loss_weight = 0.1

    model = DeepSeekV3WithMTP(
        vocab_size=vocab_size, d_model=d_model, num_layers=num_layers,
        nhead=nhead, num_mtp_heads=num_mtp_heads,
        dim_feedforward=dim_feedforward, mtp_loss_weight=mtp_loss_weight
    )
    print(f"Model created with {sum(p.numel() for p in model.parameters())/1e6:.2f}M params")

    # --- Test data ---
    batch_size = 2
    seq_len = 20
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

    # --- Forward pass and output verification ---
    outputs = model(input_ids, targets=input_ids)
    all_logits = outputs['logits_all']
    loss = outputs['loss']

    print("\nLogits shapes:")
    for i, logits in enumerate(all_logits):
        pred_type = "Main" if i == 0 else f"MTP k={i}"
        print(f"  {pred_type:10}: {list(logits.shape)}")

    print(f"\nTotal loss: {loss.item():.4f}")

verify_deepseek_v3_mtp()

Running this script produces output like the following. Notice how the **sequence length** (the middle dimension) of the logit tensors **decreases by one at each step**, confirming our causal chain is correctly implemented:

```
Model created with 2.01M params

Logits shapes:
  Main      : [2, 20, 1000]
  MTP k=1   : [2, 19, 1000]
  MTP k=2   : [2, 18, 1000]
  MTP k=3   : [2, 17, 1000]

Total loss: 109.1470
```

**Key observations:**
- The model has ~2.01M parameters (appropriate for our demonstration scale)
- Main logits: `[2, 20, 1000]` — full sequence length, vocabulary size 1000
- MTP k=1: `[2, 19, 1000]` — one fewer position (offset by 1)
- MTP k=2: `[2, 18, 1000]` — two fewer positions
- MTP k=3: `[2, 17, 1000]` — three fewer positions
- The total loss combines the main loss with the weighted MTP losses

> **Note:** The exact loss value may vary depending on PyTorch version and random initialization.

---

# Part 2: FP8 Quantization

We have now completed our exploration of Multi-Token Prediction. Now, we turn our attention to an equally important topic: **FP8 Quantization** — the technique that defines *how* the massive computations in DeepSeek are performed with incredible efficiency.

## What is Quantization?

At its core, every parameter in a large language model is just a number. By default, these numbers are stored with high precision using 32-bit floating-point (FP32). **Quantization** is the process of reducing this precision — converting parameters from a higher bit-width to a lower bit-width.

The intuition: just like a quantized image uses fewer colors but is still clearly recognizable, a quantized neural network uses fewer bits per parameter but maintains remarkably robust performance.

### Why Quantize?

The primary motivation is to solve the enormous memory and computational costs:

| Format | Bits | Memory for 70B params |
|--------|------|-----------------------|
| FP64   | 64   | 560 GB                |
| FP32   | 32   | 280 GB                |
| FP16   | 16   | 140 GB                |
| FP8    | 8    | 70 GB                 |

The benefits are twofold:
1. **Faster Training and Inference** — Smaller parameters mean less data movement from GPU memory to compute cores
2. **Accessibility** — Allows larger models to run on hardware with less memory

## Understanding Numerical Formats

Every floating-point number is represented using three parts:

- **Sign** (1 bit): Positive or negative
- **Exponent**: Determines magnitude/dynamic range
- **Mantissa** (Significand): Determines precision

### Key Formats:

| Format | Sign | Exponent | Mantissa | Total Bits | Key Property |
|--------|------|----------|----------|------------|---------------------------|
| FP32   | 1    | 8        | 23       | 32         | High precision baseline   |
| FP16   | 1    | 5        | 10       | 16         | Memory efficient          |
| BF16   | 1    | 8        | 7        | 16         | Same range as FP32        |
| INT8   | 1    | —        | 7        | 8          | No decimal precision      |
| FP8    | 1    | 4        | 3        | 8 (E4M3)   | DeepSeek's training format|

**FP8 (E4M3)** is at the heart of DeepSeek's strategy — an 8-bit floating-point format that retains a sign, exponent, and mantissa, offering a compromise between INT8's efficiency and floating-point flexibility.

## The Basic Mechanism: Scaling

The core quantization technique is **scaling**: mapping the range of original numbers onto the target range of the new format.

1. **Find Maximum Absolute Value** ($\alpha$): Scan the tensor to find the largest absolute value
2. **Calculate Scaling Factor**: $s = \text{target\_max} / \alpha$
3. **Quantize**: Multiply by the scaling factor and round: $q = \text{round}(x \cdot s)$
4. **De-quantize**: Divide by the scaling factor: $\hat{x} = q / s$

Let's implement this basic mechanism:

In [ ]:
def basic_quantize(tensor: torch.Tensor, target_max: float = 127.0):
    """
    Demonstrates the basic scaling mechanism for quantization.
    Maps a floating-point tensor to a target integer range.
    """
    # Step 1: Find the maximum absolute value (alpha)
    alpha = tensor.abs().max()

    # Step 2: Calculate the scaling factor
    scale = target_max / (alpha + 1e-8)

    # Step 3: Quantize (scale and round)
    quantized = torch.round(tensor * scale).clamp(-target_max, target_max)

    return quantized, scale


def basic_dequantize(quantized: torch.Tensor, scale: float):
    """
    Recovers the approximate original values from quantized representation.
    """
    # Step 4: De-quantize (divide by scale)
    return quantized / scale


# --- Demonstration ---
original = torch.tensor([-7.59, 2.31, 10.80, -4.12, 0.55])
print(f"Original values:      {original.tolist()}")

quantized, scale = basic_quantize(original, target_max=127.0)
print(f"Quantized (INT8):     {quantized.tolist()}")
print(f"Scaling factor:       {scale.item():.4f}")

recovered = basic_dequantize(quantized, scale)
print(f"De-quantized values:  {recovered.tolist()}")
print(f"Quantization error:   {(original - recovered).abs().tolist()}")

## The Five Pillars of DeepSeek's FP8 Training

DeepSeek's approach is not a single technique but a sophisticated framework composed of **five key innovations**:

1. **The Mixed Precision Framework** — Strategic use of different numerical formats for different operations
2. **Fine-Grained Quantization** — Block-wise scaling instead of per-tensor scaling
3. **Increasing Accumulation Precision** — Higher precision for matrix multiplication accumulators
4. **Mantissa Over Exponents (E4M3)** — Choosing precision over range for training stability
5. **Online Quantization** — Computing scaling factors from the current batch, not delayed statistics

Let's explore each of these pillars.

### Pillar 1: The Mixed Precision Framework

The central insight: **not all operations require the same level of precision**.

DeepSeek's framework strategically uses different formats for different parts of training:

| Operation | Input Precision | Compute Precision | Output/Storage |
|-----------|----------------|-------------------|----------------|
| Forward (Fprop) | FP8 | FP8 × FP8 → FP32 accumulator | BF16 |
| Dgrad (∂L/∂x) | FP8 | FP8 × FP8 → FP32 accumulator | BF16 |
| Wgrad (∂L/∂W) | FP8 | FP8 × FP8 → FP32 accumulator | FP32 |
| Weight Update | FP32 | FP32 | FP32 |

**Key decisions:**
- All three major GEMM operations use FP8 for speed (up to 2× over BF16)
- Master weights and optimizer states stay in FP32 for stability
- Weight gradients are stored in FP32 (most sensitive)
- Inter-layer activations use BF16 as a robust middle ground

**Sensitive modules kept in BF16** (bypassing FP8):
- Embedding modules
- Final output head
- MoE gating modules
- Normalization layers
- Attention softmax and context vector calculation

### Pillar 2: Fine-Grained Quantization

Standard per-tensor quantization uses a **single scaling factor** for an entire matrix. This is problematic when a tensor has both very large and very small values — the small values lose precision.

DeepSeek's solution: **tile-wise quantization**. Instead of one scale per tensor, they compute separate scaling factors for small blocks (tiles) of the tensor:

- For **activations**: 1×128 tiles (per-token, groups of 128 features)
- For **weights**: 128×128 tiles (square blocks)

This ensures that outlier values in one part of the tensor don't destroy precision elsewhere.

In [ ]:
def per_tensor_quantize(tensor: torch.Tensor, target_max: float = 448.0):
    """
    Standard per-tensor quantization: one scale factor for the entire tensor.
    target_max=448.0 corresponds to FP8 E4M3 max value.
    """
    alpha = tensor.abs().max()
    scale = target_max / (alpha + 1e-8)
    quantized = (tensor * scale).clamp(-target_max, target_max)
    dequantized = quantized / scale
    return dequantized, scale


def fine_grained_quantize(tensor: torch.Tensor, tile_size: int = 128, target_max: float = 448.0):
    """
    DeepSeek's fine-grained (tile-wise) quantization.
    Computes separate scaling factors for each tile of the tensor.
    """
    rows, cols = tensor.shape
    # Pad if necessary
    pad_cols = (tile_size - cols % tile_size) % tile_size
    if pad_cols > 0:
        tensor_padded = F.pad(tensor, (0, pad_cols))
    else:
        tensor_padded = tensor

    # Reshape into tiles of [rows, num_tiles, tile_size]
    num_tiles = tensor_padded.shape[1] // tile_size
    tiles = tensor_padded.view(rows, num_tiles, tile_size)

    # Compute per-tile scaling factors
    alpha_per_tile = tiles.abs().amax(dim=-1, keepdim=True)  # [rows, num_tiles, 1]
    scales = target_max / (alpha_per_tile + 1e-8)

    # Quantize each tile independently
    quantized_tiles = (tiles * scales).clamp(-target_max, target_max)

    # Dequantize
    dequantized_tiles = quantized_tiles / scales
    dequantized = dequantized_tiles.view(rows, -1)[:, :cols]

    return dequantized, scales


# --- Comparison: Per-Tensor vs Fine-Grained ---
torch.manual_seed(42)

# Create a tensor with outliers in one region
tensor = torch.randn(4, 256)
tensor[:, :64] *= 100  # Large values in first 64 columns
tensor[:, 64:] *= 0.01  # Small values in remaining columns

deq_per_tensor, _ = per_tensor_quantize(tensor)
deq_fine_grained, _ = fine_grained_quantize(tensor, tile_size=128)

error_per_tensor = (tensor - deq_per_tensor).abs().mean()
error_fine_grained = (tensor - deq_fine_grained).abs().mean()

print(f"Per-tensor quantization error:    {error_per_tensor.item():.6f}")
print(f"Fine-grained quantization error:  {error_fine_grained.item():.6f}")
print(f"Error reduction:                  {error_per_tensor / error_fine_grained:.2f}x")

### Pillar 3: Increasing Accumulation Precision

When multiplying two FP8 matrices, the intermediate results (partial sums) need to be accumulated. On NVIDIA H800 GPUs, the hardware accumulator for FP8 operations defaults to limited precision.

DeepSeek addresses this by promoting partial sums to **FP32** at regular intervals during the matrix multiplication. This ensures that even though the inputs are in FP8, the accumulated result maintains high fidelity.

### Pillar 4: Mantissa Over Exponents (E4M3 vs E5M2)

FP8 comes in two flavors:
- **E4M3**: 4 exponent bits, 3 mantissa bits → Higher precision, smaller range
- **E5M2**: 5 exponent bits, 2 mantissa bits → Lower precision, larger range

DeepSeek chose **E4M3 for both forward and backward passes**. The extra mantissa bit provides higher precision, which is critical for training stability. This choice is complemented by fine-grained quantization, which handles any range issues through tile-wise scaling.

### Pillar 5: Online Quantization

Traditional approaches use **delayed scaling**: the scaling factor for the current step is based on the maximum value observed in **previous** steps. This can cause issues when the data distribution shifts between steps.

DeepSeek uses **online quantization**: the scaling factor is computed from the **current** tensor being quantized. This is more computationally expensive (requires an extra pass to find the max) but eliminates any staleness in the scaling factors.

In [ ]:
def online_quantize(tensor: torch.Tensor, target_max: float = 448.0):
    """
    Online quantization: compute scale from the CURRENT tensor.
    DeepSeek's approach — always uses fresh statistics.
    """
    alpha = tensor.abs().max()
    scale = target_max / (alpha + 1e-8)
    quantized = (tensor * scale).clamp(-target_max, target_max)
    return quantized, scale


class DelayedScalingQuantizer:
    """
    Delayed scaling: uses the max value from the PREVIOUS step.
    Traditional approach that can become stale.
    """
    def __init__(self, target_max: float = 448.0):
        self.target_max = target_max
        self.prev_alpha = None

    def quantize(self, tensor: torch.Tensor):
        if self.prev_alpha is None:
            # First step: fall back to online
            alpha = tensor.abs().max()
        else:
            # Use previous step's max (delayed)
            alpha = self.prev_alpha

        scale = self.target_max / (alpha + 1e-8)
        quantized = (tensor * scale).clamp(-self.target_max, self.target_max)

        # Store current max for next step
        self.prev_alpha = tensor.abs().max().detach()

        return quantized, scale


# --- Comparison: Online vs Delayed Scaling ---
torch.manual_seed(42)

delayed_quantizer = DelayedScalingQuantizer()

print("Simulating 5 steps with changing data distributions:\n")
for step in range(5):
    # Simulate shifting distributions
    magnitude = 10 ** (step - 2)  # 0.01, 0.1, 1.0, 10.0, 100.0
    tensor = torch.randn(64, 64) * magnitude

    # Online: always uses current statistics
    q_online, s_online = online_quantize(tensor)
    deq_online = q_online / s_online
    err_online = (tensor - deq_online).abs().mean()

    # Delayed: uses previous step's statistics
    q_delayed, s_delayed = delayed_quantizer.quantize(tensor)
    deq_delayed = q_delayed / s_delayed
    err_delayed = (tensor - deq_delayed).abs().mean()

    print(f"Step {step} (magnitude={magnitude:>6.2f}): "
          f"Online err={err_online.item():.6f}, "
          f"Delayed err={err_delayed.item():.6f}")

## Putting It All Together: FP8 Linear Layer

Now let's combine several of these pillars into a practical FP8 linear layer that demonstrates DeepSeek's approach. This implementation brings together:

- **Fine-grained (tile-wise) quantization** for both activations and weights
- **Online quantization** with fresh scaling factors
- **FP32 accumulation** for the matrix multiplication result
- **Mixed precision storage** (master weights in FP32, computation in simulated FP8)

In [ ]:
class FP8Linear(nn.Module):
    """
    A linear layer that simulates DeepSeek's FP8 mixed precision framework.

    Combines:
    - Fine-grained tile-wise quantization
    - Online scaling factors
    - FP32 master weights with FP8 compute
    """
    def __init__(self, in_features: int, out_features: int, tile_size: int = 128):
        super().__init__()
        # Master weights stored in full precision (FP32)
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.02)
        self.bias = nn.Parameter(torch.zeros(out_features))
        self.tile_size = tile_size
        self.fp8_max = 448.0  # E4M3 max value

    def _quantize_tile_wise(self, tensor: torch.Tensor) -> tuple:
        """
        Apply fine-grained quantization with per-tile scaling.
        """
        orig_shape = tensor.shape
        # Flatten to 2D for tile processing
        if tensor.dim() > 2:
            tensor_2d = tensor.reshape(-1, tensor.shape[-1])
        else:
            tensor_2d = tensor

        rows, cols = tensor_2d.shape
        pad_cols = (self.tile_size - cols % self.tile_size) % self.tile_size
        if pad_cols > 0:
            tensor_padded = F.pad(tensor_2d, (0, pad_cols))
        else:
            tensor_padded = tensor_2d

        # Reshape into tiles
        num_tiles = tensor_padded.shape[1] // self.tile_size
        tiles = tensor_padded.view(rows, num_tiles, self.tile_size)

        # Online scaling: compute from current values
        alpha = tiles.abs().amax(dim=-1, keepdim=True)
        scales = self.fp8_max / (alpha + 1e-8)

        # Quantize and immediately dequantize (simulate FP8 rounding)
        quantized = torch.round(tiles * scales).clamp(-self.fp8_max, self.fp8_max)
        dequantized = (quantized / scales).view(rows, -1)[:, :cols]

        if len(orig_shape) > 2:
            dequantized = dequantized.reshape(orig_shape)

        return dequantized

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Quantize activations and weights to simulated FP8
        x_fp8 = self._quantize_tile_wise(x)
        w_fp8 = self._quantize_tile_wise(self.weight)

        # Matrix multiplication (accumulated in FP32)
        output = F.linear(x_fp8, w_fp8, self.bias)

        return output


# --- Test the FP8 Linear Layer ---
fp8_layer = FP8Linear(256, 512, tile_size=128)
std_layer = nn.Linear(256, 512)

# Copy weights for fair comparison
with torch.no_grad():
    std_layer.weight.copy_(fp8_layer.weight)
    std_layer.bias.copy_(fp8_layer.bias)

test_input = torch.randn(2, 16, 256)
out_fp8 = fp8_layer(test_input)
out_std = std_layer(test_input)

print(f"FP8 output shape:  {list(out_fp8.shape)}")
print(f"Std output shape:  {list(out_std.shape)}")
print(f"Mean abs error:    {(out_fp8 - out_std).abs().mean().item():.6f}")
print(f"Relative error:    {((out_fp8 - out_std).abs() / (out_std.abs() + 1e-8)).mean().item():.4%}")

## Conclusion

In this chapter, we have implemented the two key innovations that power DeepSeek's training efficiency:

### Multi-Token Prediction
- Built a complete **causal MTP architecture** with a sequential chain of MTP modules
- Each module merges the previous hidden state with the next token's embedding, creating a powerful forecasting mechanism
- The combined loss provides **denser training signals** and **better planning at choice points**

### FP8 Quantization
- Explored the **five pillars** of DeepSeek's FP8 framework:
  1. Mixed Precision — strategic use of FP8, BF16, and FP32
  2. Fine-Grained Quantization — tile-wise scaling for better precision
  3. Increased Accumulation Precision — FP32 accumulators
  4. E4M3 format — prioritizing mantissa over exponent
  5. Online Quantization — fresh scaling factors from current data

Together, these techniques allow DeepSeek to train massive 671B-parameter models at a fraction of the cost of competitors, completing Stage 3 of our journey. The next chapters will cover the training pipeline, post-training alignment, and knowledge distillation.